---
title: "Tell Metro"
subtitle: "From Excel to Python"
execute:
  echo: true
---

**In this lesson**

- The memo's job: one page, the decision first, the evidence behind it
- Collecting the findings: every "Back to the decision" in one table
- Choosing: the three tests a finding has to pass to reach the first page
- The chart on the first page, built the chapter 6 way
- The memo itself
- The appendix: the table, the methods, the caveats, the ask
- Presenting it, and the ten questions a skeptical reader will ask
- Critique: how to read someone else's memo, including your own

**The one idea**

> The memo is the decision the work supports, the three facts behind it, and one chart, on one page, followed by everything a skeptic needs to check it. Most of what you did stays out, and sorting the work into what goes in and what stays out is how you find the three facts.

::: {.content-visible when-format="html"}
[Download this notebook](chapter_10_lecture.ipynb){download="chapter_10_lecture.ipynb"} &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/lshpaner/excel_to_python_exercises/blob/main/notebooks/chapter_10/chapter_10_lecture.ipynb){.colab-link}
:::

::: {.content-visible unless-format="html"}
Notebook and exercises for this chapter: `github.com/lshpaner/excel_to_python_exercises`, folder `notebooks/chapter_10/`.
:::

## Back to the decision, for the last time

Nine chapters ago the board said ridership was falling and asked what Metro should do. Every chapter since has ended by returning to that question with one more piece of evidence. This chapter puts the pieces on one page.

The page has a reader, and the reader is not you. A board member will give the memo four minutes, will read the first paragraph and look at the chart, and will decide in that time whether to read the rest. So the memo is written backward from that: the recommendation first, the evidence for it second, the chart beside them, and the methods, caveats and data requests in an appendix for the one reader in ten who wants them. Consultants call this the pyramid [@minto2009]; the rest of the chapter is the pyramid applied to nine chapters of evidence. Nothing about the order in which the work was done survives into the memo.

In [ ]:
try:
    import eda_toolkit
except ImportError:                      # Colab does not have it yet; local setups install it from requirements.txt
    %pip install -q eda_toolkit
    import eda_toolkit

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import statsmodels.formula.api as smf
from eda_toolkit import generate_table1

LOCAL = "../../data/raw/"
URL = ("https://github.com/lshpaner/excel_to_python_exercises/"
       "releases/download/data/")
src = LOCAL if os.path.exists(LOCAL + "metro_trips.csv") else URL

raw = pd.read_csv(src + "metro_trips.csv")
stations = pd.read_csv(src + "metro_stations.csv", encoding="cp1252").dropna(how="all")
stations.columns = [c.strip() for c in stations.columns]
stations["Kiosk ID"] = stations["Kiosk ID"].astype(int)

raw["start_time"] = pd.to_datetime(raw["start_time"])
trips = raw[(raw["duration"] > 1) & (raw["duration"] < 1440)].copy()      # chapter 3's keep mask
trips["date"] = trips["start_time"].dt.normalize()
trips["weekend"] = trips["start_time"].dt.weekday >= 5
region = stations[["Kiosk ID", "Region"]].rename(columns={"Kiosk ID": "start_station", "Region": "region"})
trips = trips.merge(region, on="start_station", how="left")
trips["region"] = trips["region"].fillna("(no region)")

BLUE, ORANGE, LIGHT, MID = "#1F4E79", "#C55A11", "#9DB4CC", "#5B86B5"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams.update({"font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
                     "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "figure.titlesize": 11})
len(raw), len(trips)

The setup is every earlier chapter's setup at once: load, clean with the chapter 3 mask, join the station table from chapter 3, add the weekend flag from chapter 5. One notebook now reproduces every number in the memo, which is the first thing a careful reader will ask for and the reason the appendix can be short.

## Collecting the findings

Before choosing what goes on the page, put every candidate in one place, with the number, where it came from, and how sure it is. This cell recomputes each one from the clean table so that the memo and the notebook cannot drift apart.

In [ ]:
per_day = trips.groupby("date").size()
by_region = trips[trips["region"] != "(no region)"].groupby("region").agg(trips=("trip_id", "size"), stations=("start_station", "nunique"))
by_region["share"] = by_region["trips"] / by_region["trips"].sum()
by_region["per_station"] = by_region["trips"] / by_region["stations"]
pass_by_region = pd.crosstab(trips["region"], trips["passholder_type"], normalize="index")
day = per_day.rename("trips").reset_index()
day["day_num"] = (day["date"] - day["date"].min()).dt.days
day["weekday"] = day["date"].dt.strftime("%a")
trend = smf.ols("trips ~ day_num + C(weekday)", data=day).fit()
electric_week = trips.groupby(((trips["date"] - trips["date"].min()).dt.days // 7))["bike_type"].apply(lambda s: (s == "electric").mean())

findings = pd.DataFrame([
    ["1", "Typical trip is 15 minutes (median); the 29-minute mean is pulled up by a few hundred multi-hour trips",
     f"median {trips['duration'].median():.0f}, mean {trips['duration'].mean():.1f}", "certain (85,434 trips)"],
    ["3", "4% of raw rows are undocked or capped and were removed before every number below",
     f"{len(raw) - len(trips):,} of {len(raw):,} rows", "certain"],
    ["2, 5", "Monthly Pass holders take three trips in five; pay-per-ride riders (Walk-up, One Day) take three in ten",
     f"Monthly Pass {(trips['passholder_type'] == 'Monthly Pass').mean():.0%}; Walk-up and One Day Pass {trips['passholder_type'].isin(['Walk-up', 'One Day Pass']).mean():.0%}", "certain"],
    ["5", "DTLA is three fifths of trips; the Westside is busier per station",
     f"DTLA {by_region.loc['DTLA', 'share']:.0%}, {by_region.loc['DTLA', 'per_station']:.0f}/station; Westside {by_region.loc['Westside', 'share']:.0%}, {by_region.loc['Westside', 'per_station']:.0f}/station", "certain"],
    ["5, 10", "DTLA is a pass-holder system; the Westside is half pay-per-ride",
     f"Monthly Pass share DTLA {pass_by_region.loc['DTLA', 'Monthly Pass']:.0%}, Westside {pass_by_region.loc['Westside', 'Monthly Pass']:.0%}", "certain"],
    ["6, 7", "Weekend trips are longer and weekend days are busier",
     f"median {trips.loc[trips['weekend'], 'duration'].median():.0f} vs {trips.loc[~trips['weekend'], 'duration'].median():.0f} min (p < 0.001); {per_day[per_day.index.weekday >= 5].mean():,.0f} vs {per_day[per_day.index.weekday < 5].mean():,.0f} trips/day (p = 0.07)", "trips certain; days not settled"],
    ["8", "Inside the quarter ridership rose, not fell",
     f"+{trend.params['day_num'] * 7:.0f} trips/day per week, 95% CI {trend.conf_int().loc['day_num', 0] * 7:.0f} to {trend.conf_int().loc['day_num', 1] * 7:.0f}, weekday held fixed", "likely; one quarter only"],
    ["8", "Electric share fell steadily across the quarter",
     f"{electric_week.iloc[0]:.0%} in week 1 to {electric_week.iloc[-1]:.0%} in week 13", "certain as a trend; cause unknown"],
    ["9", "A trip's pass type is guessable from duration and bike type, to a point",
     "73% accuracy vs 61% baseline; electric bike multiplies the odds of a pass holder by 4", "certain on held-out trips"],
], columns=["chapter", "finding", "number", "how sure"])
pd.set_option("display.max_colwidth", None)
findings

Nine findings, every one computed on clean rows, which is why the per-station figures (340 and 495) sit a little under chapter 3's (353 and 516, computed before cleaning). The memo will keep three. The next section is how to choose.

## Choosing

A finding earns a place on the first page by passing three tests, in order.

1. **Does it change what Metro would do?** The median trip being 15 minutes is true, certain, and changes nothing on its own. The Westside being half pay-per-ride changes where a pass campaign goes. Only the second kind goes on the first page; the first kind goes in the appendix if it supports something there, and nowhere otherwise.
2. **Did it survive chapter 7?** The weekend gap in trip length survives any test. The weekend gap in daily counts does not (p = 0.07, 26 weekend days), so it cannot be stated as a fact on the first page. It can be stated as an open question in the appendix, with the n beside it.
3. **Can it be said in one sentence with a number in it?** If the sentence needs a chart to be understood, the chart goes on the first page and the sentence is its title. If it needs a paragraph, it is not a first-page finding yet.

Run the nine through the tests and three survive, and they happen to answer the board's question in order: the premise, the structure, the lever.

- The premise is wrong for this quarter: ridership rose about 12 trips a day per week, with weekdays held fixed, and the quarter cannot speak to the year.
- The system is two systems: DTLA is three-quarters pass holders on short weekday trips; the Westside is half pay-per-ride riders on longer weekend trips, and busier per station than DTLA.
- The lever is the bike: electric bikes carry 57% of pass-holder trips and 24% of Walk-up trips, and their share of all trips fell from 51% to 37% across the quarter, which nothing in the trip data explains.

Everything else is appendix material or cut.

## The chart on the first page

Chapter 6's method: write the sentence, pick the chart that makes a reader believe it fastest, put the sentence in the title. The sentence is the second finding, and the chart that carries it is the pass mix by region, built by hand so that every mark earns its place.

In [ ]:
#| label: fig-memo
#| fig-cap: "The memo's first-page chart. One bar per region, the pass mix inside it, the finding in the title, the values on the bars, and nothing else."
order = ["Monthly Pass", "Annual Pass", "One Day Pass", "Walk-up"]
colors = {"Monthly Pass": BLUE, "Annual Pass": MID, "One Day Pass": LIGHT, "Walk-up": ORANGE}
mix = pass_by_region.loc[["DTLA", "Westside", "North Hollywood"], order]
n_by_region = by_region.loc[mix.index, "trips"]

fig, ax = plt.subplots(figsize=(9, 3.6))
left = np.zeros(len(mix))
for col in order:
    vals = mix[col].values
    ax.barh(mix.index, vals, left=left, color=colors[col], label=col, height=0.6)
    for i, (v, l) in enumerate(zip(vals, left)):
        if v > 0.06:
            ax.text(l + v / 2, i, f"{v:.0%}", ha="center", va="center",
                    color="white" if col in ("Monthly Pass", "Walk-up") else "black")
    left += vals
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_yticks(range(len(mix)))
ax.set_yticklabels([f"{r}\n{n:,} trips" for r, n in zip(mix.index, n_by_region)])
ax.invert_yaxis()
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.set_title("DTLA is a pass-holder system; the Westside and North Hollywood are half pay-per-ride", loc="left")
plt.tight_layout()

What is not on it: a legend title, gridlines, a y-axis label, the "(no region)" station with six trips, and the per-station numbers, which are a sentence in the text rather than a second axis on the chart. The trip counts under the region names are there so that no reader mistakes North Hollywood's bar for a region the size of the others.

## The memo

::: {.callout-note icon=false title="MEMORANDUM"}
**To:** Metro Bike Share board
**From:** Analytics
**Re:** Q1 2026 ridership: what the trip data says, and what to do next

**Recommendation.** Treat the first quarter as two systems and manage them separately: hold the downtown pass product steady, and put the next quarter's effort into the Westside, where half of trips are pay-per-ride and the stations are the busiest in the network. Before any decision about the electric fleet, reconcile the trip data with the fleet's service records, because the electric share of trips fell by a quarter over three months and the trips alone cannot say why.

**Three findings.**

1. *Ridership did not fall this quarter.* Daily trips rose by about 12 a week from January to March (95% interval 2 to 21), with weekdays held fixed. One quarter cannot speak to the year; four quarters of daily counts with a weather column would.
2. *Downtown and the Westside are different businesses.* DTLA is 60% of trips and 75% of them are Monthly Pass holders on short weekday rides. The Westside is 38% of trips, half of them pay-per-ride, longer and heavier on weekends, and its 66 stations each see 495 starts a quarter against DTLA's 340.
3. *The electric bike is the pass holder's bike, and it is disappearing from the data.* Electric bikes carry 57% of Monthly Pass trips and 24% of Walk-up trips. Their share of all trips fell from 51% in the first week to 37% in the last. Fleet availability, not rider preference, is the likelier cause, and only the service records can tell.

**What we would do with the next quarter.** Target the pass offer at pay-per-ride riders whose trips look like a pass holder's (short, weekday, electric); the model in the appendix finds about 3,400 such trips in Q1, most of them Annual Pass and Walk-up riders on the Westside. Ask operations for the weekly count of electric bikes in service. Add a weather feed to the daily table.

*Appendix: data and cleaning, methods, the comparison the quarter could not settle, and the full table by region.*
:::

One page. The recommendation is the first word after the header, each finding is one sentence of claim and one of number, and the asks are concrete enough to be assigned to someone. The hedges are in the memo, not stripped from it: "one quarter cannot speak to the year" and "the trips alone cannot say why" are the two sentences a skeptic would otherwise supply, and it is better that the memo supplies them first.

## The appendix

The appendix is for the reader who wants to check. It has four parts and none of them is long.

**Data and cleaning.** One paragraph: 88,932 trips from Metro's Q1 2026 file; 3,498 removed (undocked bikes at one minute or less, and trips at the 24-hour cap) leaving 85,434; stations joined from Metro's station table; one station with six trips has no region. Every number in the memo is from the clean table.

**Methods.** One line per method, naming the chapter 7 to 9 tool: the weekend comparison is Welch's t-test and Mann-Whitney on trips; the trend is a regression of daily trips on day of quarter with weekday dummies; the pass-type model is a logistic regression scored on a held-out 20% of trips.

**The comparison the quarter could not settle.** Weekend days averaged 1,010 trips against 925 on weekdays, a 9% gap, p = 0.07 with 26 weekend days. This is reported as unsettled rather than as a finding, and it is the reason for the four-quarters request.

**The table.** Chapter 7's Table 1, by region, so that every comparison in the memo has its n and its p-value in one place. `generate_table1` wants exactly two groups, and the memo's argument is DTLA against the Westside, so the table is those two.

In [ ]:
two = trips[trips["region"].isin(["DTLA", "Westside"])].copy()
two["pay_per_ride"] = two["passholder_type"].isin(["Walk-up", "One Day Pass"]).astype(int)
two["electric"] = (two["bike_type"] == "electric").astype(int)
two["weekend"] = two["weekend"].astype(int)
generate_table1(two[["duration", "pay_per_ride", "electric", "weekend", "region"]],
                continuous_cols=["duration"], categorical_cols=["pay_per_ride", "electric", "weekend"],
                groupby_col="region", value_counts=True, decimal_places=1,
                drop_columns=["Mode", "Missing (n)", "Missing (%)", "Min", "Max"])

Every p-value in the table is zero to any precision, which is what 80,000 trips do to any real difference, and the table is in the appendix rather than on the first page for exactly that reason: it proves the differences are not noise; the sizes in the memo are what make them matter.

## Presenting it

The memo will be read, and then it will be presented, and the presentation is the memo read aloud in a different order. Open with the recommendation. Show the chart and say its title. Give the three findings one breath each. Stop, and take the questions, because the questions are the point of the meeting and the memo exists so that they can be short.

Ten questions a skeptical reader will ask, and where the memo answers each:

| The question | Where the answer is |
|:--|:--|
| How many trips is this based on? | First line of the appendix |
| What did you throw out, and why? | Data and cleaning |
| Is the trend real or is it weather? | Finding 1's interval, and the weather-feed ask |
| Could it be one bad week? | Trend was fitted to all 90 days with weekdays held fixed |
| Is 12 trips a week a lot? | Against 949 a day: about 1% a week |
| Why the Westside and not DTLA? | The chart; 495 against 340 starts per station |
| Are weekends really busier? | The unsettled comparison, reported as such |
| Did the electric share fall because riders chose standard bikes? | The fleet-records ask: the data cannot tell |
| How good is the pass-type model? | 73% against a 61% baseline, in the methods line |
| What would change your mind? | Four quarters, a weather column, the fleet count |

Prepare for the last question most carefully; the answer is the three data requests. A memo that cannot say what would change its mind is an opinion.

## Critique

The chapter ends where the exercises start: reading a memo as its most skeptical reader. The checks are the same ones the book has been applying since chapter 1, in the order a reader applies them.

1. **Is the recommendation the first thing on the page?** If the reader has to find it, the memo is a report.
2. **Does each first-page number have an n and an uncertainty, or a reason it does not need one?** Chapter 7.
3. **Is any first-page finding a comparison that was not tested, or a trend fitted to a sample too short to carry it?** Chapters 7 and 8.
4. **Does the chart's title state the finding, and would the finding survive if the chart were deleted?** Chapter 6. If the chart is decoration, cut it; if the finding needs it, keep it and cut the sentence.
5. **Is the mean reported where the median was the right number?** Chapter 1.
6. **Is there a finding that is true, certain, and changes nothing?** Move it to the appendix.
7. **Is the cleaning stated, and would a reader who disagreed with it be able to redo the numbers?** Chapter 3.
8. **Is a prediction's accuracy reported next to its baseline?** Chapter 9.
9. **Is there a causal word ("because", "drives", "leads to") on a correlational finding?** Chapter 8. The electric-share finding is the test case: the memo says the share fell and names two possible causes; it does not say riders turned away.
10. **Does the memo say what would change its mind?** If not, it is not finished.

Apply the list to the memo above and it passes nine and is weakest on the sixth: the per-station comparison is close to a finding that changes nothing, and a stricter editor would move it to the appendix and let the chart carry the Westside argument alone.

## Where this leaves you

The book started with a spreadsheet and the board's question and ends with a one-page answer and a notebook that reproduces every number in it. The handoff from Excel to Python happened somewhere in chapter 5, when the questions outgrew what a pivot table could hold, and it was not a replacement. The chapter 1 habit of looking at the column before summarizing it, the chapter 2 habit of asking "compared to what", and the chapter 3 habit of writing down what was cleaned and why are the same habits in both tools. Python is what made them scale to 85,000 rows, four tests, two models and one memo, and what made the whole chain rerunnable when the next quarter's file arrives.

That file will arrive. The memo's last paragraph is a list of what to do with it.

## Exercises

- **Worked exercise:** `chapter_10_guided.ipynb`. Rebuild the appendix: the findings table, the first-page chart with a title of your own, and the Table 1 for a different pair of groups. Then apply the ten-point critique to the memo and record where it is weakest.
- **On your own:** `chapter_10_solo.ipynb`. Write the memo for a different reader: the operations manager who runs the fleet, not the board. Same evidence, different decision, different first page. Then write the paragraph a skeptic on the board would send back, and answer it.

## Appendix: reproducing this chapter {.unnumbered}

`chapter_10_lecture.ipynb` is generated from this page; exercises come from `tools/build_chapter_10_exercises.py`. Every number in the memo is computed in this notebook from `data/raw/` with the chapter 3 `keep` mask; the trend model is `statsmodels`, the table is `eda_toolkit`'s `generate_table1`, and the first-page chart is matplotlib.